## Correlation Analysis — NIFTY 50 Stocks

### Objective
Identify highly correlated stock pairs from the NIFTY 50 universe to reduce the search space for cointegration testing.

Correlation is computed using **log returns** rather than prices to ensure stationarity.

### Steps
1. Load cleaned price dataset
2. Compute log returns
3. Compute correlation matrix
4. Visualize correlation structure
5. Extract top correlated pairs
6. Filter strong candidate pairs


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Load Cleaned Price Data

The dataset generated in the previous notebook is loaded from the processed data folder.


In [ ]:
close_prices = pd.read_csv("../data/processed/nifty50_close_prices.csv", index_col="Date", parse_dates=True)

print("Dataset shape:", close_prices.shape)
close_prices.head()
print("Number of stocks:", close_prices.shape[1])
print("Number of observations:", close_prices.shape[0])


In [ ]:
# Check for missing values
print("Missing values:", close_prices.isna().sum().sum())


### Compute Log Returns

Correlation should be computed on returns rather than prices.
Log returns are used because they are additive and more suitable for statistical modeling.


In [ ]:
log_returns = np.log(close_prices / close_prices.shift(1))
log_returns = log_returns.dropna()

log_returns.head()

In [ ]:
corr_matrix = log_returns.corr()

In [ ]:
plt.figure(figsize=(16,12))

sns.heatmap(
    corr_matrix,
    cmap="coolwarm",
    center=0,
    square=True,
    cbar_kws={"shrink":0.8},
    xticklabels=False,
    yticklabels=False
)

plt.title("Correlation Matrix of NIFTY 50 Stocks")
plt.savefig("../reports/correlation_matrix.png", dpi=300, bbox_inches="tight")
plt.show()


### Extract Unique Stock Pairs

The correlation matrix contains duplicate information.
We extract only the **upper triangular portion** to obtain unique stock pairs and then sort them by correlation strength.


In [ ]:
corr_pairs = (
    corr_matrix
    .where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    .stack()
    .sort_values(ascending=False)
)

top_pairs = corr_pairs.head(10)

print("\nTop 10 Correlated Stock Pairs\n")
print(top_pairs)


### Select Strongly Correlated Pairs

Pairs with correlation greater than **0.75** are considered strong candidates for cointegration testing in the next notebook.


In [ ]:
strong_pairs = corr_pairs[corr_pairs > 0.75]

print("Strong correlation pairs:")
print(strong_pairs.head(20))

In [ ]:
strong_pairs.to_csv("../data/processed/high_correlation_pairs.csv",header=["correlation"])

print("Strong correlation pairs saved successfully")


### Output

The notebook produces:

• Correlation matrix visualization  
• Top correlated stock pairs  
• Strong candidate pairs for cointegration testing  

The strong pairs dataset is saved to:

data/processed/high_correlation_pairs.csv
